# Tema 08 — Operaciones de entrada-salida

**Programación Python · Laboratorio Guiado**

Este notebook reproduce el laboratorio en pasos progresivos. Ejecuta las celdas en orden para mantener las variables y ficheros generados por cada ejercicio.

## Laboratorio 7. Ejercicio integrado: normalizar CSV y exportar a JSON

**Objetivo:** Procesar un CSV de inventario de empresa, normalizar datos, validar puertos, separar registros rechazados y exportar el resultado a JSON.

### 1. Comprobar `inventario_empresa.csv`

Debe existir el fichero `Tema08/data/inventario_empresa.csv`. Será el CSV empresarial de entrada.

In [ ]:
from pathlib import Path

carpeta = Path("../data")
carpeta.mkdir(parents=True, exist_ok=True)
entrada_csv = carpeta / "inventario_empresa.csv"
print("Existe inventario_empresa.csv:", entrada_csv.exists())
if entrada_csv.exists():
    print(entrada_csv.read_text(encoding="utf-8"))


### 2. Comprobar el CSV de entrada

Abrir el fichero `07_ejercicio_integrado_csv_a_json.py`. Definir rutas de entrada y salida.

In [ ]:
import csv
import json
from pathlib import Path

carpeta = Path("../data")
carpeta.mkdir(parents=True, exist_ok=True)
entrada_csv = carpeta / "inventario_empresa.csv"
salida_json = carpeta / "inventario_empresa_normalizado.json"
salida_rechazados = carpeta / "inventario_empresa_rechazados.csv"

print("=== 1. Comprobar fichero de entrada ===")
if not entrada_csv.exists():
    raise FileNotFoundError(f"No existe el fichero de entrada: {entrada_csv}")
print("Entrada:", entrada_csv)


### 3. Definir funciones auxiliares

Crear funciones para normalizar texto, validar puertos y normalizar una fila completa del CSV.

In [ ]:
print("\n=== 2. Funciones auxiliares ===")
def normalizar_texto(valor):
    # Limpia espacios y convierte texto a minúsculas.
    return valor.strip().lower()

def validar_puerto(puerto_txt):
    # Convierte y valida un puerto.
    puerto = int(puerto_txt)
    if not 1 <= puerto <= 65535:
        raise ValueError("Puerto fuera de rango")
    return puerto

def normalizar_fila(fila):
    # Normaliza una fila del CSV de inventario.
    return {
        "departamento": normalizar_texto(fila["departamento"]),
        "servidor": normalizar_texto(fila["servidor"]),
        "servicio": normalizar_texto(fila["servicio"]),
        "puerto": validar_puerto(fila["puerto"].strip()),
        "estado": normalizar_texto(fila["estado"]),
        "criticidad": normalizar_texto(fila["criticidad"]),
    }

print("Funciones definidas.")


### 4. Leer, normalizar y validar CSV

Separar registros válidos y rechazados sin detener todo el proceso por una fila defectuosa.

In [ ]:
print("\n=== 3. Leer, normalizar y validar CSV ===")
registros_validos = []
registros_rechazados = []
with open(entrada_csv, "r", newline="", encoding="utf-8") as filecsv:
    lector = csv.DictReader(filecsv)
    for numero_linea, fila in enumerate(lector, start=2):
        try:
            normalizada = normalizar_fila(fila)
        except ValueError as error:
            rechazada = dict(fila)
            rechazada["linea"] = numero_linea
            rechazada["motivo"] = str(error)
            registros_rechazados.append(rechazada)
        else:
            registros_validos.append(normalizada)

print("Registros válidos:", len(registros_validos))
print("Registros rechazados:", len(registros_rechazados))


### 5. Construir la estructura JSON final

Crear un diccionario con metadatos y la lista de servicios válidos normalizados.

In [ ]:
print("\n=== 4. Construir estructura JSON final ===")
salida = {
    "origen": str(entrada_csv),
    "total_validos": len(registros_validos),
    "total_rechazados": len(registros_rechazados),
    "servicios": registros_validos,
}
print(json.dumps(salida, indent=4, ensure_ascii=False))


### 6. Escribir el JSON normalizado

Guardar la estructura final en `inventario_empresa_normalizado.json`.

In [ ]:
print("\n=== 5. Escribir JSON normalizado ===")
with open(salida_json, "w", encoding="utf-8") as filejson:
    json.dump(salida, filejson, indent=4, ensure_ascii=False)
print("JSON generado:", salida_json)


### 7. Escribir CSV de registros rechazados

Guardar las filas rechazadas con el número de línea y el motivo del rechazo.

In [ ]:
print("\n=== 6. Escribir CSV de rechazados ===")
campos_rechazados = ["linea", "departamento", "servidor", "servicio", "puerto", "estado", "criticidad", "motivo"]
with open(salida_rechazados, "w", newline="", encoding="utf-8") as filecsv:
    escritor = csv.DictWriter(filecsv, fieldnames=campos_rechazados)
    escritor.writeheader()
    escritor.writerows(registros_rechazados)
print("CSV de rechazados generado:", salida_rechazados)


### 8. Comprobar el JSON generado

Leer de nuevo el JSON y mostrar un resumen de los servicios exportados.

In [ ]:
print("\n=== 7. Leer el JSON generado para comprobarlo ===")
with open(salida_json, "r", encoding="utf-8") as filejson:
    comprobacion = json.load(filejson)

for servicio in comprobacion["servicios"]:
    print(f"{servicio['servidor']} -> {servicio['servicio']}:{servicio['puerto']} [{servicio['estado']}]")


### 9. Ejecución del script completo

```bash
python 07_ejercicio_integrado_csv_a_json.py
```

**Resultado esperado:** el alumno debe integrar rutas, CSV, JSON, validación, normalización y gestión de registros rechazados en un proceso de datos empresarial.